In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

## Summarizarion in Middleware

Automatically summarise conversation history when approaching token limit,preserving recent messages while compressing
older context.Summarizarion is useful fro the following
- Long-running conversations that exeecn context windows.
- Multi-turn dialogue with extensive history
- Applications where preserving lengthy extensive full conversations matters

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

agent = create_agent(
        model="gpt-5",
        checkpointer=InMemorySaver(),
        middleware=[
            SummarizationMiddleware(
                model="gpt-5",
                trigger=("messages",10),
                keep=("messages",4)
            )
        ]
)

In [9]:
### Run with a Thread ID
config = {"configurable":{"thread_id":"test-1"}}

In [11]:
questions = [
    "What is 2+2?",
    "What is 2*2?",
    "What is 10/2?",
    "What is 27-2?",
    "What is 3*3?",
    "What is 2201*2?",
    "What is 8+2?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages:{len(response['messages'])}")
    

Messages: {'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\nUser is asking basic arithmetic questions and expects concise numeric answers.\n\n## SUMMARY\n- The assistant answered previous arithmetic queries:\n  - 2+2 = 4\n  - 2*2 = 4\n  - 10/2 = 5\n- The user’s latest question is pending: “What is 27-2?”\n- Strategy: Provide direct, succinct numeric results without extra explanation.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\n- Answer the pending question: 27-2 = 25.\n- Continue responding to subsequent arithmetic questions concisely.', additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='f8941dca-a96e-4fd1-94e0-419ec5ce714e'), AIMessage(content='25', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 67, 'total_tokens': 77, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_t